# Overview
- Task: Model Inference for Evaluation
- Model: Qwen3.5-4B
- Test set:

# Setup

In [ ]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install unsloth
# !pip install evaluate

import os
# before import torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True" # Prevents memory fragmentation and OOM
import torch
import unsloth
from unsloth import FastLanguageModel
from datasets import load_from_disk, Dataset, load_dataset
import pandas as pd

# Dataset for inference

In [1]:
dataset_path = "/kaggle/input/datasets/ilovesamir/dial2note-additional-dataset/test_set_participant_version.csv"
ds = pd.read_csv(dataset_path)
ds = Dataset.from_dict(ds)
print(f"Structure of the dataset:\n{ds}")
dialogues = ds['dialogue']

print("✅ Test dataset ready!")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


NameError: name 'pd' is not defined

# Model loading

In [ ]:
adapter_path = "/kaggle/input/datasets/ilovesamir/qwen3-5-4b/qwen3.5-4B_SFT"
max_seq_length = 2048

# # 1. model + LoRA adapter (manual integration)
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = base_model_name,
#     max_seq_length = max_seq_length,
#     dtype = None,
#     load_in_4bit = True,
#     full_finetuning = False,
#     random_state = 42,
# )
# model.load_adapter(adapter_path)
# print("Model + adapter successful")


# 2. automatic integraion (unsloth)
# base_model_path given in configuration file
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_path,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
    full_finetuning = False,
    random_state = 42,
)
print(""✅ The base model and the adapter have been merged successfully")

# Inference (Generation)

In [ ]:
# The instruction used in training
instruction = (
"Convert the medical dialogue into a SOAP note "
"(Subjective, Objective, Assessment, Plan). "
"Use only the provided information.\n"
)

# example_dialogue = """
# [Doctor]: Hello there. What brings you in today?

# [Patient]: Hi, doctor. Two days ago, I tripped while going down the stairs, and since then, my right ankle has been really painful and swollen.

# [Doctor]: I’m sorry to hear that. Does the pain get worse when you try to walk?

# [Patient]: Yes, it’s not too bad when I’m resting, but as soon as I put weight on it or try to walk, the pain is much sharper.

# [Doctor]: I see. Let me take a look at the ankle. (After a brief examination) I see some bruising and swelling around the outer ankle bone here. Does it hurt when I press this spot?

# [Patient]: Ow! Yes, that really hurts. Do you think it’s broken?

# [Doctor]: It’s hard to tell for sure right now. Your vitals—blood pressure and temperature—look normal, but because you have tenderness and swelling over the lateral malleolus, we need to rule out a fracture.

# [Patient]: Okay. I haven't had any fever or anything, but the pain is the main issue. What’s the plan?

# [Doctor]: First, I’ll prescribe Ibuprofen 400mg for the pain. You can take it every 6 hours as needed. Also, I’m ordering a 3-view X-ray of your right ankle to get a better look at the bone.

# [Patient]: That makes sense. Should I come back here after the X-ray?

# [Doctor]: Exactly. Once the X-ray results are ready, we’ll review them together and decide on the next steps for your treatment.
# """

# example_note = (
#     "1. **Subjective:**\n\n"
#     "**Chief Complaint (CC):**\n"
#     "- Right ankle pain and swelling.\n\n"
#     "**History of Present Illness (HPI):**\n"
#     "- The patient presents with right ankle pain and swelling following a trip on the stairs 2 days ago. The pain is described as sharp upon weight-bearing or walking and improves with rest. There is no history of fever or similar prior injuries.\n\n"
#     "**Review of Systems (ROS):**\n"
#     "- Musculoskeletal: Positive for right ankle pain, swelling, and bruising.\n"
#     "- General: Negative for fever or chills.\n"
#     "- Cardiovascular: No complaints.\n"
#     "- Respiratory: No complaints.\n\n"
#     "2. **Objective:**\n\n"
#     "**Vital Signs:**\n"
#     "- BP: 120/80 mmHg\n"
#     "- HR: 72 bpm\n"
#     "- RR: 18 breaths/min\n"
#     "- Temp: 98.6°F\n"
#     "- SpO2: 98% on room air\n\n"
#     "**Physical Examination:**\n"
#     "- Musculoskeletal: Bruising and swelling noted around the outer ankle. Tenderness localized over the right lateral malleolus.\n\n"
#     "3. **Assessment:**\n\n"
#     "**Diagnosis:**\n"
#     "- Acute right ankle sprain. Rule out lateral malleolus fracture due to significant tenderness and swelling.\n\n"
#     "4. **Plan:**\n\n"
#     "**Medical Management:**\n"
#     "- Ibuprofen 400 mg orally every 6 hours as needed for pain.\n\n"
#     "**Investigations Ordered:**\n"
#     "- X-ray of the right ankle (3 views) to rule out fracture.\n\n"
#     "**Follow-Up:**\n"
#     "- The patient is instructed to return to the clinic for a review of X-ray results and to determine further treatment steps."
# )

# FIXME : 프롬프트 마지막에 '마침표' 유도
# 프롬프트가 <|im_start|>assistant\n<think>\n...\n</think>\n로 끝난다면,
# 모델은 그 뒤에 나올 내용이 무궁무진하다고 생각합니다. 하지만 훈련 데이터가 항상 특정 형식으로 끝난다면, 그 끝을 명시해주는 것이 좋습니다.
# 가장 효과적인 방법은 학습 데이터 마지막에 유니크한 종료 문구를 넣는 것입니다.
# 예: 모든 학습 데이터 마지막에 [End of SOAP Note]를 넣고 학습시킨 뒤, 추론 시 stop_strings=["[End of SOAP Note]"]를 사용.

def get_predictions(model, tokenizer, dialogue_list, batch_size=8):
    FastLanguageModel.for_inference(model)
    all_predictions = []
    # few_shot_user = f"Dialogue:\n{example_dialogue}"
    # few_shot_assistant = example_note
    # anchored_text = "1. **Subjective:**\n\n**Chief Complaint (CC):**\n-"
    for i in range(0, len(dialogue_list), batch_size):
        batch_texts = dialogue_list[i : i + batch_size]
        prompts = []
        for d in batch_texts:
            messages = [
                {"role": "system", "content": instruction},
    
                # Few-shot
                # {"role": "user", "content": few_shot_user},
                # {"role": "assistant", "content": few_shot_assistant},
    
                {"role": "user", "content": f"Dialogue:\n{d}"}
            ]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            # prompt += anchored_text
            prompts.append(prompt)
        inputs = tokenizer(text=prompts, padding=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id, # <|vision_pad|>
                eos_token_id=tokenizer.eos_token_id, # <|im_end|>
                # repetition_penalty=1.15,
                # no_repeat_ngram_size=3,
            )
        
        input_len = inputs.input_ids.shape[1]
        final_outputs = outputs[:, input_len:]
        
        decoded_outputs = tokenizer.batch_decode(final_outputs, skip_special_tokens=True)
        all_predictions.extend([p.strip() for p in decoded_outputs])
        # for p in decoded_outputs:
        #   full_note = anchored_text + p
        #   all_predictions.append(full_note.strip())
    return all_predictions

In [ ]:
predictions = get_predictions(model, tokenizer, dialogues, batch_size=8)
print("✅ Finished generation")

# Evaluation

In [ ]:
# import evaluate

# # Loading the metrics
# rouge_metric = evaluate.load("rouge")
# bleu_metric = evaluate.load("bleu")
# meteor_metric = evaluate.load("meteor")

# def calculate_all_metrics(predictions, references):
#     # ROUGE-1, ROUGE-2, ROUGE-L
#     rouge_results = rouge_metric.compute(
#         predictions=predictions, 
#         references=references, 
#         use_stemmer=True
#     )
    
#     # BLEU
#     bleu_results = bleu_metric.compute(
#         predictions=predictions, 
#         references=references
#     )
        
#     # METEOR
#     meteor_results = meteor_metric.compute(
#         predictions=predictions, 
#         references=references
#     )
    
#     print("\n" + "="*40)
#     print("📋 Final Performance Metrics")
#     print("="*40)
#     print(f"🔹 BLEU     : {bleu_results['bleu']:.4f}")
#     print(f"🔹 ROUGE-1 : {rouge_results['rouge1']:.4f}")
#     print(f"🔹 ROUGE-2 : {rouge_results['rouge2']:.4f}")
#     print(f"🔹 ROUGE-L : {rouge_results['rougeL']:.4f}")
#     print(f"🔹 METEOR  : {meteor_results['meteor']:.4f}")
#     print("="*40)

# calculate_all_metrics(predictions, references)

# print("Check the first result")
# # prediction and reference pair check
# print("Prediction: ")
# print(predictions[0])
# print("####################################################")
# print("####################################################")
# print("Reference: ")
# print(references[0])


# Saving the results

In [ ]:
df = pd.DataFrame({
    "id": ds['id'],
    "generated_note": predictions
})
df.to_csv("final_inference_result.csv", index=False, encoding="utf-8-sig")
print("✅ Results saved")